In [ ]:
%matplotlib widget
import numpy as np
import tmodel
import tensorflow as tf
from tensorflow import keras
from base_model import float_to_binary_array_not_IEEE, create_dense_model
import matplotlib.pyplot as plt

data=np.load('jordan_data.npz',allow_pickle=True)
signals = data['signals']
times = data['times']
signal = 2
instance = 0

X = times[signal].copy()
X = X/X.max()

X = np.array([float_to_binary_array_not_IEEE(X[i]) for i in range(len(X))])
Y = signals[signal]
validation_split = int(0.8*X.shape[0])

Xtrain=X[:validation_split]
Xval=X[validation_split:]
Ytrain=Y[:validation_split]
Yval=Y[validation_split:]

strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    model = create_dense_model(dropout_frac=0.5,n_streams=20)
    model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=.01), loss='mae')

model.load_weights(f"{tmodel.data_dir}/base_model_{signal}_{instance}.weights.h5")
p0=model.predict(Xtrain,batch_size=256)
p1=model.predict(Xval,batch_size=256)

plt.figure(figsize=(15,5))
plt.plot(times[signal],Y,label='truth')
plt.plot(times[signal][:validation_split],p0[:,0],label='train prediction')
plt.plot(times[signal][validation_split:],p1[:,0],label='val prediction')

plt.show()